# Beta Translator

Local workflow: video → poses → reviewed holds → wall alignment → contact drafts →
confirmed endpoints → target-body IK. The final replay shows the original tracked motion;
the IK result is one static adapted pose.

Open this notebook from the repository folder using its `.venv` Python kernel.
For first-time setup, run `bash setup.sh` in a terminal, then select `.venv/bin/python`.
The notebook checks dependencies and reads local files; it does not install packages.
After changing a helper `.py` file, restart the kernel and rerun the notebook.

Pose tracking uses MediaPipe; hold detection uses the pretrained
[Indoor Climbing Hold and Route Segmentation](https://github.com/xiaoxiae/Indoor-Climbing-Hold-and-Route-Segmentation)
model. Credit these sources in the report. MediaPipe sample code copyright 2023 The MediaPipe Authors,
Apache License 2.0.

## 1. Local settings and imports

Set the video and output folder here. Keep each video's reviewed files in its own output folder.
`USE_SAVED_HOLD_MAP = True` reuses reviewed IDs; set it to `False` for a new detection.
With `REUSE_POSES = True`, poses are reused only when the video, pose model, and settings match.
`RATIOS` is defined once in the IK module. Override individual entries below if you have measurements.

Generated files are grouped under `outputs/videos/`, `outputs/tables/`, `outputs/images/`,
`outputs/state/`, and `outputs/exports/`. The source video stays in the repository's `videos/` folder.


In [ ]:
from pathlib import Path
import importlib.util
import json
import sys

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'beta_translator').is_dir():
    raise FileNotFoundError('Open this notebook from the Beta_Translator repository folder.')
MODELS_DIR = PROJECT_DIR / 'models'
VIDEO_FILE = str(PROJECT_DIR / 'videos/trial1.mp4')
OUTPUTS_DIR = PROJECT_DIR / 'outputs'
VIDEO_OUTPUT_DIR = OUTPUTS_DIR / 'videos'
TABLES_DIR = OUTPUTS_DIR / 'tables'
IMAGES_DIR = OUTPUTS_DIR / 'images'
STATE_DIR = OUTPUTS_DIR / 'state'
EXPORTS_DIR = OUTPUTS_DIR / 'exports'
USE_SAVED_HOLD_MAP = True
REUSE_POSES = True
TARGET_HEIGHT_CM = 165

required = ['cv2', 'mediapipe', 'numpy', 'pandas', 'matplotlib', 'scipy', 'ipywidgets']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError(f'Missing {missing}. Run bash setup.sh, then select the .venv kernel.')

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mediapipe as mp
import ipywidgets as widgets
from IPython.display import Video, display
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from beta_translator.inverse_kinematics import RATIOS, LIMBS, make_body, forward_kinematics, solve_inverse_kinematics
from beta_translator.pose_adapter import (
    LIMB_LANDMARKS, to_frame0, image_to_wall_cm, wall_cm_to_image,
    pose_row_to_vector, angle_visibility_weights,
)
from beta_translator.contact_matching import match_limb_contacts
from beta_translator.project_io import (
    file_sha256, read_frame, find_ffmpeg, convert_video, load_reviewed_holds,
    restore_reviewed_map, save_reviewed_holds, load_alignment, save_alignment,
    load_annotations, save_annotations,
)

# Optional measured proportions, using the same RATIOS name:
# RATIOS = {**RATIOS, 'forearm': measured_forearm_cm / TARGET_HEIGHT_CM}
POSE_SETTINGS = dict(min_pose_detection_confidence=0.5,
                     min_pose_presence_confidence=0.5, min_tracking_confidence=0.5)
POSE_MODEL = MODELS_DIR / 'pose_landmarker.task'
for path in (Path(VIDEO_FILE), POSE_MODEL):
    if not path.is_file() or path.stat().st_size < 1024:
        raise FileNotFoundError(f'Missing asset or Git LFS pointer: {path}')
if not USE_SAVED_HOLD_MAP:
    for name in ('experiment_config.yml', 'model_final.pth'):
        if not (MODELS_DIR / name).is_file():
            raise FileNotFoundError(f'Missing model asset: {MODELS_DIR / name}')
    if importlib.util.find_spec('detectron2') is None:
        raise ImportError('Fresh hold detection requires Detectron2; run bash setup.sh first.')
find_ffmpeg()
for folder in (VIDEO_OUTPUT_DIR, TABLES_DIR, IMAGES_DIR, STATE_DIR, EXPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
VIDEO_HASH = file_sha256(VIDEO_FILE)
PX_PER_CM = np.hypot(80, -21) / 46 if VIDEO_HASH == 'db656ed38a93a619f6df7dbd65804718860aa9191e650a1a92952ab6063a007a' else None
# For another video, replace None with its measured pixel distance / distance in cm.
print('Python:', sys.executable)
print('Video:', VIDEO_FILE)
print('Outputs:', OUTPUTS_DIR)


## 2. Track poses once and save both outputs

The annotated video and coordinate table now come from the same MediaPipe results and settings.
Missing detections leave gaps in the table; frame numbers retain the original video timing.
Visibility is an uncertainty indicator, not a measured joint-position error.

In [ ]:
# Copyright 2023 The MediaPipe Authors. All Rights Reserved.
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

def draw_landmarks_on_image(rgb_image, detection_result):
    pose_landmarks_list = detection_result.pose_landmarks
    annotated_image = np.copy(rgb_image)

    height, width, _ = annotated_image.shape

    for pose_landmarks in pose_landmarks_list:

        # Draw every landmark
        for landmark in pose_landmarks:
            x = int(landmark.x * width)
            y = int(landmark.y * height)

            if 0 <= x < width and 0 <= y < height:
                cv2.circle(
                    annotated_image,
                    (x, y),
                    5,
                    (255, 255, 0),
                    -1
                )

        # Draw every connection
        for connection in vision.PoseLandmarksConnections.POSE_LANDMARKS:
            start = pose_landmarks[connection.start]
            end = pose_landmarks[connection.end]

            x1 = int(start.x * width)
            y1 = int(start.y * height)

            x2 = int(end.x * width)
            y2 = int(end.y * height)

            cv2.line(
                annotated_image,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

    return annotated_image


In [ ]:
important_landmarks = {
    'left_shoulder': 11, 'right_shoulder': 12, 'left_elbow': 13, 'right_elbow': 14,
    'left_wrist': 15, 'right_wrist': 16, 'left_hip': 23, 'right_hip': 24,
    'left_knee': 25, 'right_knee': 26, 'left_ankle': 27, 'right_ankle': 28,
    'left_heel': 29, 'right_heel': 30, 'left_foot': 31, 'right_foot': 32,
}
POSE_PATH = TABLES_DIR / 'pose_coordinates.csv'
POSE_MANIFEST = STATE_DIR / 'pose_manifest.json'
pose_inputs = dict(video_sha256=VIDEO_HASH, model_sha256=file_sha256(POSE_MODEL), settings=POSE_SETTINGS)


def track_video():
    options = vision.PoseLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path=str(POSE_MODEL)),
        running_mode=vision.RunningMode.VIDEO, num_poses=1, **POSE_SETTINGS)
    cap = cv2.VideoCapture(VIDEO_FILE)
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    width, height = (int(cap.get(prop)) for prop in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT))
    if not np.isfinite(fps) or fps <= 0 or min(width, height) <= 0:
        cap.release()
        raise ValueError('Could not read video metadata.')
    writer = cv2.VideoWriter(str(VIDEO_OUTPUT_DIR / 'pose_output.mp4'),
                            cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    if not writer.isOpened():
        cap.release()
        raise RuntimeError('Could not create the pose video.')
    rows, frame_index = [], 0
    try:
        with vision.PoseLandmarker.create_from_options(options) as detector:
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                timestamp_ms = round(frame_index * 1000 / fps)
                result = detector.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), timestamp_ms)
                writer.write(cv2.cvtColor(draw_landmarks_on_image(rgb, result), cv2.COLOR_RGB2BGR))
                if result.pose_landmarks:
                    row = dict(frame=frame_index, time_sec=frame_index/fps, time_ms=timestamp_ms)
                    for name, index in important_landmarks.items():
                        point = result.pose_landmarks[0][index]
                        row.update({f'{name}_x_norm': point.x, f'{name}_y_norm': point.y,
                                    f'{name}_z': point.z, f'{name}_x_px': point.x * width,
                                    f'{name}_y_px': point.y * height, f'{name}_visibility': point.visibility})
                    rows.append(row)
                frame_index += 1
    finally:
        cap.release()
        writer.release()
    if not rows:
        raise ValueError('No poses detected. Check the video before continuing.')
    poses = pd.DataFrame(rows)
    poses.to_csv(POSE_PATH, index=False)
    manifest = dict(inputs=pose_inputs, pose_sha256=file_sha256(POSE_PATH),
                    overlay_sha256=file_sha256(VIDEO_OUTPUT_DIR / 'pose_output.mp4'))
    POSE_MANIFEST.write_text(json.dumps(manifest, indent=2))
    return poses


saved_pose = json.loads(POSE_MANIFEST.read_text()) if POSE_MANIFEST.is_file() else {}
can_reuse = (REUSE_POSES and POSE_PATH.is_file() and (VIDEO_OUTPUT_DIR / 'pose_output.mp4').is_file()
             and saved_pose.get('inputs') == pose_inputs
             and saved_pose.get('pose_sha256') == file_sha256(POSE_PATH)
             and saved_pose.get('overlay_sha256') == file_sha256(VIDEO_OUTPUT_DIR / 'pose_output.mp4'))
pose_df = pd.read_csv(POSE_PATH) if can_reuse else track_video()
print('Reused matching pose outputs.' if can_reuse else 'Generated coordinates and video from one tracking pass.')
print('Frames with poses:', len(pose_df))
display(pose_df.head())


## 3. Preview the tracked skeleton

Inspect wrists, feet, hips, and shoulders. H.264 conversion uses the local FFmpeg installation.

In [ ]:
convert_video(VIDEO_OUTPUT_DIR / 'pose_output.mp4', VIDEO_OUTPUT_DIR / 'pose_output_h264.mp4')
display(Video(str(VIDEO_OUTPUT_DIR / 'pose_output_h264.mp4'), embed=True, width=360))


## 4. Load or detect holds, then review the route

All holds use **frame 0**, the same reference used for wall alignment and IK.
Saved maps preserve IDs and verify the input video. If a saved map belongs to another video,
choose a new output folder and disable `USE_SAVED_HOLD_MAP`.

For fresh detection, place the matching `experiment_config.yml` and `model_final.pth` in `models/`.
They come from the authors' `Basic_Augmented_P2P3_DoubleTopK_NMS_2KIT`
[model dataset](https://www.kaggle.com/datasets/tomasslama/indoor-climbing-gym-hold-segmentation).
Review missed holds, false detections, and mask boundaries.

In [ ]:
HOLD_MAP_PATH = STATE_DIR / 'reviewed_hold_map.npz'
if USE_SAVED_HOLD_MAP and not HOLD_MAP_PATH.is_file():
    restore_reviewed_map(PROJECT_DIR / 'data/reviewed_contact_data.zip', HOLD_MAP_PATH, VIDEO_FILE)

# Use the first frame of the current video as the reference.
reference_cap = cv2.VideoCapture(VIDEO_FILE)
ok, reference_frame = reference_cap.read()
reference_cap.release()
if not ok:
    raise ValueError(f'Could not read the first frame of {VIDEO_FILE}.')
wall_image = reference_frame.copy()
print('Reference taken from:', VIDEO_FILE)
print('Reference size:', wall_image.shape[:2])
cv2.imwrite(str(IMAGES_DIR / 'wall_reference.jpg'), wall_image)

if USE_SAVED_HOLD_MAP:
    candidate_records, candidate_masks, candidate_run_id, saved_selected_hold_ids = load_reviewed_holds(
        HOLD_MAP_PATH, VIDEO_FILE)
    print(f'Loaded reviewed hold map: {len(candidate_masks)} fixed IDs')
else:
    from detectron2.config import get_cfg
    from detectron2.engine import DefaultPredictor

    hold_cfg = get_cfg()
    hold_cfg.merge_from_file(str(MODELS_DIR / 'experiment_config.yml'))
    hold_cfg.MODEL.WEIGHTS = str(MODELS_DIR / 'model_final.pth')
    # CPU is slower but avoids requiring a GPU build for this single image.
    hold_cfg.MODEL.DEVICE = 'cpu'
    SCORE_THRESHOLD = 0.70
    hold_cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = SCORE_THRESHOLD
    hold_predictor = DefaultPredictor(hold_cfg)
    hold_instances = hold_predictor(wall_image)['instances'].to('cpu')
    if not hold_instances.has('pred_masks'):
        raise ValueError('The loaded model did not return instance masks. Check the model/config pair.')

def build_hold_candidates(masks, scores, classes):
    """Keep pixel geometry at original image size and assign deterministic IDs per run."""
    unsorted = []
    for model_index, (mask, score, class_id) in enumerate(zip(masks, scores, classes)):
        ys, xs = np.nonzero(mask)
        if len(xs) == 0:
            continue
        unsorted.append((float(ys.mean()), float(xs.mean()), model_index, {
            'model_index': int(model_index), 'class_id': int(class_id),
            'class_name': {0: 'hold', 1: 'volume'}.get(int(class_id), 'other'),
            'score': float(score), 'cx_px': float(xs.mean()), 'cy_px': float(ys.mean()),
            'x_min': int(xs.min()), 'y_min': int(ys.min()),
            'x_max': int(xs.max()), 'y_max': int(ys.max()), 'area_px': int(len(xs)),
            'source': 'model_prediction', 'review_status': 'unreviewed'
        }))
    records, mask_map = [], {}
    for i, (_, _, model_index, record) in enumerate(sorted(unsorted, key=lambda x: x[:3]), 1):
        record['hold_id'] = f'H{i:02d}'
        records.append(record)
        mask_map[record['hold_id']] = np.asarray(masks[model_index], dtype=bool)
    return records, mask_map

import uuid
if not USE_SAVED_HOLD_MAP:
    candidate_run_id = uuid.uuid4().hex
    candidate_records, candidate_masks = build_hold_candidates(
        hold_instances.pred_masks.numpy(), hold_instances.scores.numpy(),
        hold_instances.pred_classes.numpy())
if not candidate_records:
    raise ValueError('No candidates. Inspect the reference image and try a lower threshold.')

def render_holds(frame, ids, H=None):
    if frame.shape[:2] != wall_image.shape[:2]:
        raise ValueError('Video and reference image dimensions differ. Do not mix resized coordinates.')
    result = frame.copy()
    h, w = frame.shape[:2]
    for hold_id in ids:
        record = next(r for r in candidate_records if r['hold_id'] == hold_id)
        mask = candidate_masks[hold_id]
        if H is not None:
            mask = cv2.warpPerspective(mask.astype(np.uint8), H, (w, h),
                                       flags=cv2.INTER_NEAREST).astype(bool)
            if not mask.any():
                continue
            ys, xs = np.nonzero(mask)
            xy = (int(xs.mean()), int(ys.mean()))
        else:
            xy = (int(record['cx_px']), int(record['cy_px']))
        result[mask] = (0.6 * result[mask] + 0.4 * np.array([0, 210, 255])).astype(np.uint8)
        contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(result, contours, -1, (0, 255, 255), 2)
        cv2.putText(result, hold_id, xy, cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 0), 4)
        cv2.putText(result, hold_id, xy, cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 1)
    return result

plt.figure(figsize=(10, 14))
plt.imshow(cv2.cvtColor(render_holds(wall_image, candidate_masks), cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title('Hold map: verify the IDs needed for the route')
plt.show()
display(pd.DataFrame(candidate_records))
print('Saved maps retain their IDs. Review a new detection before changing the route.')


Select all supports and destination holds needed for the movement. Saved selections are restored;
for a fresh detection, fill in `NEW_ROUTE_HOLD_IDS` using the displayed map. Saving preserves this map's IDs.

In [ ]:
# Route IDs belong to this reviewed map.
NEW_ROUTE_HOLD_IDS = []  # For a fresh detection, enter IDs from the displayed map.
SELECTED_HOLD_IDS = list(saved_selected_hold_ids) if USE_SAVED_HOLD_MAP else NEW_ROUTE_HOLD_IDS
if not SELECTED_HOLD_IDS:
    raise ValueError('Enter the relevant predicted hold IDs in SELECTED_HOLD_IDS, then rerun.')
if len(set(SELECTED_HOLD_IDS)) != len(SELECTED_HOLD_IDS):
    raise ValueError('Remove duplicate hold IDs.')
invalid = set(SELECTED_HOLD_IDS) - set(candidate_masks)
if invalid:
    raise ValueError(f'Unknown hold IDs: {invalid}')
selected_hold_ids = list(SELECTED_HOLD_IDS)
reviewed_run_id = candidate_run_id
reviewed_records = [dict(r, review_status='accepted') for r in candidate_records if r['hold_id'] in selected_hold_ids]
plt.figure(figsize=(8, 12))
plt.imshow(cv2.cvtColor(render_holds(wall_image, selected_hold_ids), cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

save_reviewed_holds(HOLD_MAP_PATH, VIDEO_FILE, wall_image, candidate_records,
                    candidate_masks, candidate_run_id, selected_hold_ids)
print('Saved route:', selected_hold_ids)


## 5. Align the wall across frames

SIFT matches and RANSAC estimate a frame-0-to-frame-i homography. Unreliable frames reuse the
previous transform and are excluded from contact inference and IK.
The cache is tied to the video, exact pose CSV, and alignment settings; changed inputs trigger recomputation.

In [ ]:
sift = cv2.SIFT_create(nfeatures=4000)
bf = cv2.BFMatcher(cv2.NORM_L2)

# Optional: polygon (in frame-0 pixels) around the flat wall face with the route,
# e.g. np.array([[x, y], [x, y], ...]). None = use the whole frame.
WALL_ROI = None

# Builds a mask that blacks out the climber
def climber_exclusion_mask(shape, frame_idx, pad=60):
    # 255 = usable wall pixels, 0 = box around the climber
    mask = np.full(shape[:2], 255, np.uint8)
    row = pose_df[pose_df.frame == frame_idx]
    if len(row):
        r = row.iloc[0]
        xs = [r[c] for c in r.index if c.endswith('_x_px')]
        ys = [r[c] for c in r.index if c.endswith('_y_px')]
        mask[max(0, int(min(ys)) - pad):int(max(ys)) + pad,
             max(0, int(min(xs)) - pad):int(max(xs)) + pad] = 0
    return mask

# Finds SIFT features in one frame and matches them to frame-0 features
def estimate_H_ref_to_frame(ref_kp, ref_des, frame_bgr, frame_idx, min_inliers=25):
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    kp, des = sift.detectAndCompute(gray, climber_exclusion_mask(gray.shape, frame_idx))
    if des is None or len(kp) < 10:
        return None, 0
    if ref_des is None:
        return None, 0
    pairs = bf.knnMatch(ref_des, des, k=2)
    good = [p[0] for p in pairs if len(p) == 2 and p[0].distance < 0.75 * p[1].distance]
    if len(good) < min_inliers:
        return None, len(good)
    src = np.float32([ref_kp[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([kp[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    H, inliers = cv2.findHomography(src, dst, cv2.RANSAC, 4.0)
    if H is None or not np.isfinite(H).all():
        return None, 0
    n = int(inliers.sum())
    return (H if n >= min_inliers else None), n

# Features in the reference frame (frame 0), excluding the climber and anything outside the ROI.
ref_gray = cv2.cvtColor(wall_image, cv2.COLOR_BGR2GRAY)
ref_mask = climber_exclusion_mask(ref_gray.shape, 0)
if WALL_ROI is not None:
    roi = np.zeros_like(ref_mask)
    cv2.fillPoly(roi, [WALL_ROI.astype(np.int32)], 255)
    ref_mask = cv2.bitwise_and(ref_mask, roi)
ref_kp, ref_des = sift.detectAndCompute(ref_gray, ref_mask)

ALIGNMENT_PATH = STATE_DIR / 'alignment_cache.npz'
alignment_settings = dict(nfeatures=4000, min_inliers=25, ratio_test=0.75,
                          ransac_threshold=4.0, exclusion_pad=60,
                          wall_roi=None if WALL_ROI is None else WALL_ROI.tolist())
homographies = load_alignment(ALIGNMENT_PATH, VIDEO_FILE, POSE_PATH, alignment_settings)
if homographies is None:
    # Get homography for each frame
    homographies = {}          # frame index -> (H, inlier_count, 'ok' | 'reused')
    last_H = np.eye(3)
    cap = cv2.VideoCapture(VIDEO_FILE)
    i = 0
    while True:
        ok, f = cap.read()
        if not ok:
            break
        H, n = estimate_H_ref_to_frame(ref_kp, ref_des, f, i)
        if H is None:
            H, status = last_H, 'reused'   # fall back to the previous frame's transform
        else:
            last_H, status = H, 'ok'
        homographies[i] = (H, n, status)
        i += 1
    cap.release()
    
    if not homographies:
        raise ValueError('No video frames were available for alignment.')
    save_alignment(ALIGNMENT_PATH, homographies, VIDEO_FILE, POSE_PATH, alignment_settings)
else:
    print('Loaded matching alignment cache.')
bad = [k for k, v in homographies.items() if v[2] != 'ok']
print(f'{len(homographies)} frames; {len(bad)} reused alignments.')


## 6. Draft limb contacts

The matcher compares wrists and foot-index landmarks with selected hold masks in each video frame.
A unique nearby hold needs 0.5 seconds of consecutive visible evidence to become `likely_contact`.
Ambiguity, low visibility, missing poses, and unreliable alignment remain unresolved.
`no_candidate` does not mean a free limb. Margins are pixels at 720px video width, scaled to the input.

This adapts the proximity and temporal filtering idea described in
[The Way Up](https://arxiv.org/html/2505.12854v1), using masks instead of boxes.
The score describes geometric proximity, not a contact probability. It does not establish grip or load bearing.

In [ ]:
if reviewed_run_id != candidate_run_id:
    raise ValueError('Review the current hold map first.')
CONTACT_SETTINGS = dict(hand_margin_px=20, foot_margin_px=10, min_visibility=0.5,
                        min_contact_sec=0.5, ambiguity_gap_px=3, min_inliers=25)
contact_run_id = reviewed_run_id
contact_hold_ids = tuple(selected_hold_ids)
contact_masks = {hold: candidate_masks[hold].copy() for hold in contact_hold_ids}
contacts_df, contact_events_df, contact_fps = match_limb_contacts(
    pose_df, contact_masks, homographies, VIDEO_FILE, **CONTACT_SETTINGS)

contacts_df.to_csv(TABLES_DIR / 'limb_contacts.csv', index=False)
contact_events_df.to_csv(TABLES_DIR / 'contact_events.csv', index=False)
(STATE_DIR / 'contact_settings.json').write_text(json.dumps(dict(
    settings=CONTACT_SETTINGS, video=str(VIDEO_FILE), video_sha256=file_sha256(VIDEO_FILE),
    pose_sha256=file_sha256(POSE_PATH), alignment_sha256=file_sha256(ALIGNMENT_PATH), fps=contact_fps,
    selected_holds=contact_hold_ids, candidate_run_id=contact_run_id,
    coordinates='original current-frame pixels', margins='pixels at 720px image width',
    interval='start_frame/end_frame inclusive; end_sec exclusive',
    match_score='geometric proximity only, not a probability'), indent=2))
display(pd.crosstab(contacts_df['limb'], contacts_df['status']))
display(contact_events_df)
print(f'Saved {len(contacts_df)} limb observations and {len(contact_events_df)} draft intervals to {TABLES_DIR}.')


## 7. Review drafts and confirm BEFORE/AFTER contacts

One viewer now shows the raw frame, hold outlines, contact predictions, and manual endpoint controls.
Inspect one clear movement. Set each endpoint, then explicitly choose its four contact labels.
Predictions never fill the manual labels automatically. Left/right refer to the climber's anatomy.

Use `unknown` for occlusion, `free` for an observed limb off the wall, and `wall_contact` for a smear.
Save the annotations before restarting; Load restores matching video/map annotations.
Changing an endpoint resets its four labels so an old frame's contacts cannot carry over silently.
Foot predictions use foot-index landmarks; IK uses ankles as an approximation, so review foot contacts carefully.
For continuous IK, enter release (first free frame) and arrival (first new-contact frame),
then confirm the other three contacts were maintained throughout. Leave both at -1 for endpoint-only work.
Changing endpoints, contact labels, or the moving limb clears this motion review. Save again after edits.


In [ ]:
CONTACT_COLORS = {'likely_contact': 'limegreen', 'tentative': 'darkorange',
                  'ambiguous': 'darkorange', 'no_candidate': 'deepskyblue'}
CONTACT_LABELS = {'left_hand': 'LH', 'right_hand': 'RH', 'left_foot': 'LF', 'right_foot': 'RF'}
contact_form_run_id = reviewed_run_id
annotation_hold_ids = tuple(selected_hold_ids)
annotation_count = int(contacts_df['frame'].max()) + 1
ANNOTATION_PATH = STATE_DIR / 'endpoint_annotations.json'


def check_review_state():
    if (contact_form_run_id != candidate_run_id or contact_run_id != candidate_run_id
            or annotation_hold_ids != tuple(selected_hold_ids) or contact_hold_ids != annotation_hold_ids):
        raise ValueError('The reviewed route changed. Rerun contact matching and this viewer.')


def plot_contact_frame(frame_index, ax=None):
    check_review_state()
    frame = read_frame(VIDEO_FILE, frame_index)
    entry = homographies.get(frame_index)
    if entry is not None and entry[2] == 'ok':
        frame = render_holds(frame, contact_hold_ids, entry[0])
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 8))
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    for row in contacts_df.loc[contacts_df.frame == frame_index].itertuples():
        if not np.isfinite([row.x_px, row.y_px]).all():
            continue
        color = CONTACT_COLORS.get(row.status, 'gray')
        margin = CONTACT_SETTINGS['hand_margin_px' if 'hand' in row.limb else 'foot_margin_px']
        radius = margin * frame.shape[1] / 720
        ax.add_patch(plt.Circle((row.x_px, row.y_px), radius, fill=False, color=color))
        hold = row.candidate_hold if pd.notna(row.candidate_hold) else '?'
        ax.annotate(f'{CONTACT_LABELS[row.limb]} {hold}', (row.x_px, row.y_px),
                    xytext=(6, -14), textcoords='offset points', color=color, fontsize=9,
                    bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', pad=1))
    ax.set_title(f'Frame {frame_index} | {frame_index/contact_fps:.2f}s')
    ax.axis('off')
    return ax


frame_slider = widgets.IntSlider(min=0, max=annotation_count-1, description='Frame',
                                 continuous_update=False, layout=widgets.Layout(width='500px'))
preview_output = widgets.Output()
action_output = widgets.Output()
chosen_info = widgets.HTML('Choose both endpoint frames.')
chosen_frames = {'before': None, 'after': None}
contact_choices = ['unknown', 'free', 'wall_contact'] + list(annotation_hold_ids)
contact_widgets = {limb: {phase: widgets.Dropdown(options=contact_choices, value='unknown',
                         layout=widgets.Layout(width='150px')) for phase in chosen_frames} for limb in LIMBS}
moving_limb = widgets.Dropdown(options=LIMBS, description='Moving limb', style={'description_width': 'initial'})
annotation_notes = widgets.Textarea(description='Notes', layout=widgets.Layout(width='500px'))
release_frame = widgets.IntText(value=-1, description='Release frame',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='240px'))
arrival_frame = widgets.IntText(value=-1, description='Arrival frame',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='240px'))
supports_confirmed = widgets.Checkbox(value=False, description='Other three contacts maintained throughout',
    indent=False, layout=widgets.Layout(width='500px'))


def clear_motion_review(change=None):
    release_frame.value = -1
    arrival_frame.value = -1
    supports_confirmed.value = False


def invalidate_support_confirmation(change=None):
    supports_confirmed.value = False


def motion_state():
    if release_frame.value == -1 and arrival_frame.value == -1 and not supports_confirmed.value:
        return None
    return dict(release_frame=release_frame.value, arrival_frame=arrival_frame.value,
                support_contacts_maintained=supports_confirmed.value)



def show_contact_preview(change=None):
    with preview_output:
        preview_output.clear_output(wait=True)
        ax = plot_contact_frame(frame_slider.value)
        plt.show()
        plt.close(ax.figure)
        display(contacts_df.loc[contacts_df.frame == frame_slider.value,
                ['limb', 'candidate_hold', 'hold_id', 'visibility', 'status']])


def choose_endpoint(phase):
    check_review_state()
    if chosen_frames[phase] != frame_slider.value:
        clear_motion_review()
        chosen_frames[phase] = frame_slider.value
        for limb in LIMBS:
            contact_widgets[limb][phase].value = 'unknown'
    chosen_info.value = f"BEFORE: {chosen_frames['before']} | AFTER: {chosen_frames['after']}"


def annotation_state():
    check_review_state()
    result = dict(video_sha256=VIDEO_HASH, candidate_run_id=contact_form_run_id,
                selected_hold_ids=list(annotation_hold_ids), frames=dict(chosen_frames),
                contacts={phase: {limb: contact_widgets[limb][phase].value for limb in LIMBS}
                          for phase in chosen_frames},
                moving_limb=moving_limb.value, notes=annotation_notes.value,
                body=dict(height_cm=TARGET_HEIGHT_CM, ratios=dict(RATIOS), px_per_cm=PX_PER_CM))
    motion = motion_state()
    if motion is not None:
        result['motion'] = motion
    return result


def save_review(_=None):
    with action_output:
        action_output.clear_output(wait=True)
        try:
            save_annotations(ANNOTATION_PATH, annotation_state(), VIDEO_FILE,
                             contact_form_run_id, annotation_hold_ids, annotation_count)
            print('Saved endpoint annotations:', ANNOTATION_PATH)
        except ValueError as error:
            print(error)


def load_review(_=None):
    global TARGET_HEIGHT_CM, RATIOS, PX_PER_CM
    with action_output:
        action_output.clear_output(wait=True)
        try:
            check_review_state()
            data = load_annotations(ANNOTATION_PATH, VIDEO_FILE, contact_form_run_id,
                                    annotation_hold_ids, annotation_count)
            chosen_frames.update(data['frames'])
            for phase in chosen_frames:
                for limb in LIMBS:
                    contact_widgets[limb][phase].value = data['contacts'][phase][limb]
            moving_limb.value = data['moving_limb']
            annotation_notes.value = data['notes']
            TARGET_HEIGHT_CM = data['body']['height_cm']
            RATIOS = data['body']['ratios']
            PX_PER_CM = data['body']['px_per_cm']
            motion = data.get('motion') or {}
            release_frame.value = motion.get('release_frame', -1)
            arrival_frame.value = motion.get('arrival_frame', -1)
            supports_confirmed.value = motion.get('support_contacts_maintained', False)
            chosen_info.value = f"BEFORE: {chosen_frames['before']} | AFTER: {chosen_frames['after']}"
            print('Loaded annotations and body/wall settings.')
        except (FileNotFoundError, ValueError) as error:
            print(error)


before_button = widgets.Button(description='Set BEFORE')
after_button = widgets.Button(description='Set AFTER')
save_button = widgets.Button(description='Save annotations')
load_button = widgets.Button(description='Load annotations')
before_button.on_click(lambda _: choose_endpoint('before'))
after_button.on_click(lambda _: choose_endpoint('after'))
save_button.on_click(save_review)
load_button.on_click(load_review)
frame_slider.observe(show_contact_preview, names='value')
moving_limb.observe(clear_motion_review, names='value')
for fields in contact_widgets.values():
    for field in fields.values():
        field.observe(clear_motion_review, names='value')
release_frame.observe(invalidate_support_confirmation, names='value')
arrival_frame.observe(invalidate_support_confirmation, names='value')
rows = [widgets.HTML('<b>Limb | BEFORE contact | AFTER contact</b>')]
for limb in LIMBS:
    rows.append(widgets.HBox([widgets.Label(limb, layout=widgets.Layout(width='110px')),
                             contact_widgets[limb]['before'], contact_widgets[limb]['after']]))
controls = widgets.VBox([frame_slider, widgets.HBox([before_button, after_button]), chosen_info,
                        moving_limb, *rows,
                        widgets.HTML('Motion: release = first free frame; arrival = first contact frame. -1 = unset.'),
                        widgets.HBox([release_frame, arrival_frame]), supports_confirmed,
                        annotation_notes, widgets.HBox([save_button, load_button]), action_output])
display(widgets.HTML('Green: likely contact · Orange: tentative/ambiguous · Blue: no candidate · Gray: unavailable'))
display(widgets.HBox([preview_output, controls]))
show_contact_preview()


## 8. Fit a target climber at a reviewed endpoint

`inverse_kinematics.py` contains the only body model and solver. It preserves segment lengths,
keeps the ±150° bend bounds, clips the initial guess, scales centimeter/radian variables by the Jacobian,
and reports nonconvergence. `pose_adapter.py` aligns joints to frame 0 and converts coordinates and angles.

Change `TARGET_HEIGHT_CM`, `RATIOS`, and `PX_PER_CM` in Section 1. Ratios are approximate defaults;
use measured segment-length/height ratios when available. The wall scale is a separate measurement.

For the supplied 65-hold map, the measured 46 cm gap runs from **H44's right edge to H40's left edge**
(formerly H45/H41). The frame-0 endpoints are approximately (418, 592.5) and (498, 571.5),
giving 1.798 pixels/cm. This applies only to that video and measurement, so it is gated by the video hash.
A different video needs its own measured scale. A uniform scale does not rectify wall perspective or body depth.

Choose `before` or `after` below. Known hold IDs and `free` are supported; unknown contacts and smears
need further annotation. The output is a static geometric fit; small error does not prove physical feasibility.

In [ ]:
def fit_endpoint(frame_index, contacts, height_cm, px_per_cm, ratios=None):
    """Fit one endpoint using approximate hold-center targets in a scaled 2D image."""
    if any(v is None or not np.isfinite(v) or v <= 0 for v in (height_cm, px_per_cm)):
        raise ValueError("Height and the measured frame-0 scale must be positive.")
    if frame_index is None or frame_index not in homographies:
        raise ValueError("Choose a valid endpoint frame first.")
    if not (contact_form_run_id == reviewed_run_id == candidate_run_id) or annotation_hold_ids != tuple(selected_hold_ids):
        raise ValueError("Review and annotate the current hold predictions before fitting.")
    if set(contacts) != set(LIMBS):
        raise ValueError("Provide a contact state for each of the four limbs.")
    if homographies[frame_index][2] != "ok":
        raise ValueError("This frame needs a verified alignment before fitting.")

    rows = pose_df.loc[pose_df["frame"] == frame_index]
    if len(rows) != 1:
        raise ValueError("The chosen frame must have exactly one pose row.")
    row = rows.iloc[0].copy()
    image_height = wall_image.shape[0]

    # Landmarks start in frame-i pixels; hold centers are already in frame-0 pixels.
    names = list(dict.fromkeys(n for triplet in LIMB_LANDMARKS.values() for n in triplet))
    points = np.array([[row[f"{n}_x_px"], row[f"{n}_y_px"]] for n in names])
    aligned = to_frame0(points, homographies, frame_index)
    if not np.isfinite(aligned).all():
        raise ValueError("Alignment produced invalid landmark coordinates.")
    for name, (x, y) in zip(names, aligned):
        row[f"{name}_x_px"], row[f"{name}_y_px"] = x, y

    holds = {r["hold_id"]: r for r in reviewed_records}
    targets = {}
    for limb in LIMBS:
        state = contacts[limb]
        if state == "free":
            targets[limb] = None
        elif state in holds:
            hold = holds[state]
            targets[limb] = image_to_wall_cm(
                [[hold["cx_px"], hold["cy_px"]]], px_per_cm, image_height,
            )[0]
        else:
            raise ValueError(f"{limb}: '{state}' needs a reviewed hold/contact point.")
    if all(target is None for target in targets.values()):
        raise ValueError("At least one known contact is needed for a wall fit.")

    body = make_body(height_cm, ratios)
    reference = pose_row_to_vector(row, px_per_cm, image_height)
    fitted, errors_cm = solve_inverse_kinematics(
        body, targets, reference,
        angle_weight=0.3 * angle_visibility_weights(row),
    )

    # Draw in the same reference frame used for the hold coordinates.
    joints = forward_kinematics(fitted, body)
    fig, ax = plt.subplots(figsize=(6, 9))
    ax.imshow(cv2.cvtColor(wall_image, cv2.COLOR_BGR2RGB))
    torso_roots = [joints.limbs[name].root for name in
                   ("left_hand", "right_hand", "right_foot", "left_foot", "left_hand")]
    torso = wall_cm_to_image(torso_roots, px_per_cm, image_height)
    ax.plot(*torso.T, "o-", color="lime", linewidth=2)
    for limb, chain in joints.limbs.items():
        pixels = wall_cm_to_image(chain, px_per_cm, image_height)
        ax.plot(*pixels.T, "o-", color="lime", linewidth=2)
        if targets[limb] is not None:
            target = wall_cm_to_image([targets[limb]], px_per_cm, image_height)[0]
            ax.plot(*target, "x", color="red", markersize=9)
    ax.set(xlim=(0, wall_image.shape[1]), ylim=(image_height, 0))
    ax.set_title(f"Frame {frame_index}: approximate 2D fit, height {height_cm:g} cm")
    ax.axis("off")
    plt.show()

    print("Distance from each fitted wrist/ankle to its hold-center target (cm):")
    print({limb: float(f'{error:.6g}') for limb, error in errors_cm.items()})
    return fitted, errors_cm


In [ ]:
PHASE = 'after'  # Or 'before'.
if PHASE not in chosen_frames:
    raise ValueError("Choose 'before' or 'after'.")
if chosen_frames[PHASE] is None:
    print('Select an endpoint and confirm its contacts in Section 7 first.')
elif PX_PER_CM is None:
    print('Enter the measured wall scale in Section 1 first.')
else:
    fitted_pose, contact_errors_cm = fit_endpoint(
        chosen_frames[PHASE],
        {limb: contact_widgets[limb][PHASE].value for limb in LIMBS},
        TARGET_HEIGHT_CM, PX_PER_CM, ratios=RATIOS)


## 8B. Fit and preview a continuous 2D candidate

In Section 7, **Load annotations** to restore the saved endpoints, body settings, and motion events.
Save any edits before running this cell. The reviewed example uses 165 cm/default ratios,
BEFORE 107, RELEASE 117, ARRIVAL 128, AFTER 129, and three maintained supports.
Frame 117 is the first free frame; frame 128 is the first new-contact frame.

This solver fits the entire interval with contact penalties and time-continuity terms.
It checks contact residuals against a numerical tolerance; this is not measurement accuracy.
It refits the endpoints jointly, so the result can differ from the separate Section 8 solutions.
The free hand follows a soft minimum-jerk straight-line guide, not a measured hand trajectory.
Low-visibility reference angles receive less weight. No force, friction, balance, collision,
or real climbing success is modeled. Missing poses or unverified alignment stop the run.

The side-by-side preview shows the actual aligned source frames and the fitted body on a neutral hold map.
The second video is explicitly slowed to one-third speed for review.
Results are saved as motion_ik_* files in videos/, tables/, state/, and images/.
Section 10 includes them only when their source data and annotations still match.


In [ ]:
from beta_translator.motion_pipeline import run_motion_analysis
from IPython.display import Image

check_review_state()
saved_motion_annotations = load_annotations(
    ANNOTATION_PATH, VIDEO_FILE, contact_form_run_id, annotation_hold_ids, annotation_count)
if saved_motion_annotations != annotation_state():
    raise ValueError('Load the saved annotations, or save your current edits, before running Section 8B.')
motion_summary = run_motion_analysis(VIDEO_FILE, OUTPUTS_DIR)
print('Motion contact timing:', motion_summary['annotation']['motion'])
print('Numerical diagnostics:', json.dumps(motion_summary['metrics'], indent=2))
display(Video(str(VIDEO_OUTPUT_DIR / 'motion_ik_h264.mp4'), embed=True, width=800))
display(Video(str(VIDEO_OUTPUT_DIR / 'motion_ik_slow_h264.mp4'), embed=True, width=800))
display(Image(filename=str(IMAGES_DIR / 'ik_motion_storyboard.png'), width=900))


## 9. Replay the original movement on the fixed hold map

This replay uses observed joints, not an adapted IK animation. Green indicates visibility ≥ 0.5;
gray indicates lower visibility. Missing poses and unreliable alignment are shown explicitly.
Timing matches the input video.

In [ ]:
import tempfile


def preview_wall_motion():
    """Replay observed joints in frame-0 pixels; this does not run IK."""
    if reviewed_run_id != candidate_run_id:
        raise ValueError("Review the current hold map first.")

    cap = cv2.VideoCapture(VIDEO_FILE)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    if not np.isfinite(fps) or fps <= 0 or frame_count <= 0:
        raise ValueError("Could not read the source video's frame rate and frame count.")

    names = list(dict.fromkeys(name for chain in LIMB_LANDMARKS.values() for name in chain))
    links = [(chain[i], chain[i + 1]) for chain in LIMB_LANDMARKS.values() for i in (0, 1)]
    links += [("left_shoulder", "right_shoulder"), ("left_hip", "right_hip"),
              ("left_shoulder", "left_hip"), ("right_shoulder", "right_hip")]
    poses = pose_df.set_index("frame")
    if not poses.index.is_unique:
        raise ValueError("Pose data must contain at most one row per video frame.")
    # A neutral hold map avoids leaving the reference-frame climber in the background.
    background = render_holds(np.full_like(wall_image, 245), selected_hold_ids)
    height, width = background.shape[:2]
    output = VIDEO_OUTPUT_DIR / "wall_motion_h264.mp4"
    output.parent.mkdir(parents=True, exist_ok=True)
    uncertain_frames = 0

    with tempfile.TemporaryDirectory() as directory:
        raw_video = str(Path(directory) / "wall_motion.mp4")
        writer = cv2.VideoWriter(raw_video, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
        if not writer.isOpened():
            raise RuntimeError("Could not create the preview video.")
        try:
            for frame_index in range(frame_count):
                frame = background.copy()
                status = "tracked pose"
                if frame_index not in poses.index:
                    status = "pose unavailable"
                elif frame_index not in homographies or homographies[frame_index][2] != "ok":
                    status = "alignment unverified"
                else:
                    row = poses.loc[frame_index]
                    points = np.array([[row[f"{name}_x_px"], row[f"{name}_y_px"]] for name in names])
                    aligned = to_frame0(points, homographies, frame_index)
                    joints = {
                        name: tuple(map(int, np.rint(point)))
                        for name, point in zip(names, aligned)
                        if np.isfinite(point).all()
                    }
                    visible = {name: row[f"{name}_visibility"] >= 0.5 for name in joints}
                    if len(joints) < len(names):
                        status = "partial pose" if joints else "pose unavailable"
                    elif not all(visible.values()):
                        status = "low visibility"
                    for start, end in links:
                        if start in joints and end in joints:
                            color = (45, 155, 35) if visible[start] and visible[end] else (150, 150, 150)
                            cv2.line(frame, joints[start], joints[end], color, 4, cv2.LINE_AA)
                    for name, point in joints.items():
                        color = (45, 155, 35) if visible[name] else (150, 150, 150)
                        cv2.circle(frame, point, 5, color, -1, cv2.LINE_AA)
                if status != "tracked pose":
                    uncertain_frames += 1
                cv2.putText(frame, f"Frame {frame_index} | {frame_index / fps:.2f}s | {status}",
                            (12, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (40, 40, 40), 2, cv2.LINE_AA)
                cv2.putText(frame, "Green: visible joints | Gray: low visibility", (12, 52),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (40, 40, 40), 1, cv2.LINE_AA)
                writer.write(frame)
        finally:
            writer.release()

        convert_video(raw_video, output)

    display(Video(str(output), embed=True, width=360))
    print(f"Saved {output}: {frame_count} frames at {fps:g} FPS; {uncertain_frames} frames with uncertain or missing joints.")


preview_wall_motion()


## 10. Save review samples and export the current run

Samples adapt to video length. The bundle includes the current pose data, matching alignment,
reviewed holds, draft results, and manually saved endpoints when available.
Save annotations in Section 7 before exporting. No automatic prediction is treated as a confirmed contact.
The original video and model weights remain separate local inputs.

The ZIP is written to `outputs/exports/` and keeps the same `tables/`, `images/`, and `state/`
subfolders inside. Restoring a reviewed hold map supports both this layout and the original flat bundle.


In [ ]:
import importlib.metadata
import platform
import zipfile

from beta_translator.motion_pipeline import current_motion_outputs

check_review_state()
motion_paths = current_motion_outputs(VIDEO_FILE, OUTPUTS_DIR)
if load_alignment(ALIGNMENT_PATH, VIDEO_FILE, POSE_PATH, alignment_settings) is None:
    raise ValueError('Alignment no longer matches the saved inputs. Rerun Section 5 onward.')
contact_settings = json.loads((STATE_DIR / 'contact_settings.json').read_text())
if (contact_settings['pose_sha256'] != file_sha256(POSE_PATH)
        or contact_settings['alignment_sha256'] != file_sha256(ALIGNMENT_PATH)):
    raise ValueError('Contact results are outdated. Rerun Section 6 onward.')

# Sample indices are valid for short clips as well as the supplied example.
sample_frames = np.unique(np.linspace(0, annotation_count-1, min(5, annotation_count), dtype=int)).tolist()
fig, axes = plt.subplots(1, len(sample_frames), figsize=(3.5 * len(sample_frames), 7))
for ax, index in zip(np.atleast_1d(axes), sample_frames):
    plot_contact_frame(index, ax)
fig.tight_layout()
fig.savefig(IMAGES_DIR / 'contact_samples.png', dpi=160, bbox_inches='tight')
plt.show()
plt.close(fig)

saved_annotations = None
if ANNOTATION_PATH.is_file():
    saved_annotations = load_annotations(ANNOTATION_PATH, VIDEO_FILE, contact_form_run_id,
                                         annotation_hold_ids, annotation_count)
if any(frame is not None for frame in chosen_frames.values()) and saved_annotations != annotation_state():
    raise ValueError('Save your current endpoint annotations in Section 7 before exporting.')

summary = dict(video_sha256=VIDEO_HASH, pose_rows=len(pose_df), limb_rows=len(contacts_df),
    hold_candidates=len(candidate_masks), selected_holds=list(contact_hold_ids),
    draft_intervals=len(contact_events_df), sample_frames=sample_frames,
    alignment_frames=len(homographies), alignment_fallbacks=sum(v[2] != 'ok' for v in homographies.values()),
    candidate_run_id=candidate_run_id, status_counts=pd.crosstab(contacts_df.limb, contacts_df.status).to_dict(),
    body=dict(height_cm=TARGET_HEIGHT_CM, ratios=dict(RATIOS), px_per_cm=PX_PER_CM),
    versions=dict(python=platform.python_version(), opencv=cv2.__version__,
                  mediapipe=importlib.metadata.version('mediapipe')),
    manual_endpoints_included=saved_annotations is not None, motion_results_included=bool(motion_paths),
    interpretation='Geometric drafts and reviewed annotations; no proof of grip or physical feasibility.')
(STATE_DIR / 'validation_summary.json').write_text(json.dumps(summary, indent=2))
export_path = EXPORTS_DIR / 'reviewed_contact_data.zip'
bundle_files = [
    HOLD_MAP_PATH, POSE_PATH, POSE_MANIFEST, ALIGNMENT_PATH,
    TABLES_DIR / 'limb_contacts.csv', TABLES_DIR / 'contact_events.csv',
    STATE_DIR / 'contact_settings.json', STATE_DIR / 'validation_summary.json',
    IMAGES_DIR / 'contact_samples.png',
]
if saved_annotations is not None:
    bundle_files.append(ANNOTATION_PATH)
bundle_files.extend(motion_paths)
with zipfile.ZipFile(export_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in bundle_files:
        archive.write(path, path.relative_to(OUTPUTS_DIR))
print('Saved local bundle:', export_path)
print('Manual endpoint annotations included:', saved_annotations is not None)
